# 🍎 Fruits-360 — Full Dataset Pipeline
**SSN College of Engineering | SSN-LMS**

Processes **all 250 classes / 173,786 images** from the Fruits-360 repository.

- **Part 1** — Image Processing (IP lab)
- **Part 2** — Deep Learning Classification (DL lab)

> Runtime → Change runtime type → **T4 GPU**

---
# PART 1 — Image Processing

### 0. Install & Imports

In [ ]:
!pip install torch torchvision matplotlib scikit-learn -q

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, random, time, warnings
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.util import random_noise
warnings.filterwarnings('ignore')
print('Imports done.')

### 1. Download Full Fruits-360 Dataset (All 250 Classes)

In [ ]:
repo_path = '/content/fruits360_repo'

# Clean clone
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
    print(f'Removed existing directory: {repo_path}')

print('Cloning full Fruits-360 repository (~2 GB, takes 3-5 min)...')
!git clone https://github.com/fruits-360/fruits-360-100x100.git {repo_path}

TRAIN_DIR       = f'{repo_path}/Training'
TEST_DIR        = f'{repo_path}/Test'
SAMPLE_IMG_PATH = f'{TRAIN_DIR}/Apple Golden 1/1_100.jpg'

if os.path.exists(TRAIN_DIR):
    all_classes = sorted(os.listdir(TRAIN_DIR))
    total_train = sum(len(list(Path(f'{TRAIN_DIR}/{c}').glob('*.jpg'))) for c in all_classes)
    total_test  = sum(len(list(Path(f'{TEST_DIR}/{c}').glob('*.jpg')))  for c in all_classes)
    print(f'✅ Classes: {len(all_classes)}')
    print(f'   Training images: {total_train:,}')
    print(f'   Test images:     {total_test:,}')
    print(f'   Total:           {total_train + total_test:,}')
else:
    print('❌ Clone failed:', os.listdir(repo_path))

### 2. Read Image

In [ ]:
img = cv2.imread(SAMPLE_IMG_PATH)
plt.imshow(img)
plt.axis('off')
plt.title('Raw Image — Apple Golden 1')
plt.show()

### 3. Read Multiple Images in a Directory

In [ ]:
folder_path = f'{TRAIN_DIR}/Apple Golden 1/'
all_files   = os.listdir(folder_path)
print('All files (first 10):', all_files[:10])

images = []
for file in all_files:
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
        curr_img = cv2.imread(os.path.join(folder_path, file))
        if curr_img is not None:
            images.append(curr_img)

print(f'Total images loaded: {len(images)}')
if images:
    plt.imshow(cv2.cvtColor(images[0], cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.title('First Image'); plt.show()

### 4. Image Properties

In [ ]:
img     = cv2.imread(SAMPLE_IMG_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb); plt.axis('off'); plt.title('Image'); plt.show()

height, width, channels = img.shape
total_pixels = height * width
print('Height:', height)
print('Width:',  width)
print('Channels:', channels)
print('Total Pixels:', total_pixels)

### 5. Single Pixel Intensity

In [ ]:
row, col  = 50, 50
pixel_rgb = img_rgb[row, col]
print('RGB Intensity:', pixel_rgb)

### 6. 5×5 Image Intensities

In [ ]:
gray   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
region = gray[50:55, 50:55]
print('5x5 Grayscale region:\n', region)

### 7. Sub-Image (Crop Region)

In [ ]:
region_rgb = img_rgb[20:60, 20:60]
plt.imshow(region_rgb); plt.axis('off'); plt.title('Region (5x5 pixels)'); plt.show()

### 8. Color Models (RGB, HSV, CMY, Grayscale, Binary)

In [ ]:
rgb       = img_rgb
hsv       = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
cmy       = 255 - img_rgb
gray      = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(12, 8))
for i,(data,cmap,title) in enumerate([
    (rgb,None,'RGB'),(hsv,None,'HSV'),(cmy,None,'CMY'),
    (gray,'gray','Grayscale'),(binary,'gray','Binary')]):
    plt.subplot(3,2,i+1); plt.imshow(data,cmap=cmap)
    plt.title(title); plt.axis('off')
plt.tight_layout(); plt.show()

img_rgb        = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_double     = img_rgb.astype(np.float64)
img_normalized = img_double / 255.0
print('Original dtype:', img.dtype, '| Double dtype:', img_double.dtype)
print('Original Min/Max:', np.min(img_rgb), np.max(img_rgb))
print('Normalized Min/Max:', np.min(img_normalized), np.max(img_normalized))

### 9. Point Processing Techniques

In [ ]:
negative           = 255 - gray
bright             = np.clip(img.astype(np.int16) + 50, 0, 255).astype(np.uint8)
contrast           = np.clip(img.astype(np.float32) * 1.5, 0, 255).astype(np.uint8)
contrast_stretched = ((gray - gray.min()) / (gray.max() - gray.min() + 1e-9) * 255).astype(np.uint8)
equalized          = cv2.equalizeHist(gray)

plt.figure(figsize=(20, 16))
items = [('Original Grayscale',gray,'gray'),('Negative',negative,'gray'),
         ('Original',img_rgb,None),('Brightness +50',cv2.cvtColor(bright,cv2.COLOR_BGR2RGB),None),
         ('Original',img_rgb,None),('Contrast x1.5',contrast,'gray'),
         ('Original',img_rgb,None),('Contrast Stretched',contrast_stretched,'gray')]
for i,(title,data,cmap) in enumerate(items):
    plt.subplot(4,2,i+1); plt.imshow(data,cmap=cmap)
    plt.title(title); plt.axis('off')
plt.tight_layout(); plt.show()

### 10. Noise Models (Gaussian, Salt & Pepper, Uniform)

In [ ]:
img      = cv2.imread(SAMPLE_IMG_PATH)
img_gray = cv2.imread(SAMPLE_IMG_PATH, cv2.IMREAD_GRAYSCALE)
img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

gaussian_noise    = (random_noise(img, mode='gaussian', mean=0, var=0.01) * 255).astype(np.uint8)
sp_noise          = (random_noise(img, mode='s&p', amount=0.05) * 255).astype(np.uint8)
uniform_noise_img = np.clip(img.astype(np.int16) + np.random.randint(-5,6,img.shape,np.int16),0,255).astype(np.uint8)

plt.figure(figsize=(12,12))
for i,(data,title) in enumerate([
    (img_rgb,'Original'),
    (cv2.cvtColor(gaussian_noise,cv2.COLOR_BGR2RGB),'Gaussian Noise'),
    (cv2.cvtColor(sp_noise,cv2.COLOR_BGR2RGB),'Salt & Pepper Noise'),
    (cv2.cvtColor(uniform_noise_img,cv2.COLOR_BGR2RGB),'Uniform Noise')]):
    plt.subplot(2,2,i+1); plt.imshow(data); plt.title(title); plt.axis('off')
plt.tight_layout(); plt.show()

### 11. Filters + Manual Convolution

In [ ]:
mean_filtered     = cv2.blur(uniform_noise_img, (3,3))
median_filtered   = cv2.medianBlur(sp_noise, 5)
gaussian_filtered = cv2.GaussianBlur(gaussian_noise, (5,5), 1.5)

def convolve2d(img, kernel):
    m,n      = img.shape
    kh,kw    = kernel.shape
    ph,pw    = kh//2, kw//2
    padded   = np.pad(img,((ph,ph),(pw,pw)),mode='constant',constant_values=0)
    output   = np.zeros((m,n))
    for i in range(m):
        for j in range(n):
            output[i,j] = np.sum(padded[i:i+kh,j:j+kw]*kernel)
    return output

kernel = np.ones((3,3))/9
print('Box filter kernel:\n', kernel)
out = convolve2d(img_gray, kernel)
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(10,4))
ax1.imshow(img_gray,cmap='gray'); ax1.set_title('Original')
ax2.imshow(out,cmap='gray');      ax2.set_title('Convolved')
plt.show()

fig,axes = plt.subplots(3,2,figsize=(15,10))
pairs = [(cv2.cvtColor(gaussian_noise,cv2.COLOR_BGR2RGB),'Gaussian Noise',
          cv2.cvtColor(gaussian_filtered,cv2.COLOR_BGR2RGB),'Gaussian Filter'),
         (cv2.cvtColor(uniform_noise_img,cv2.COLOR_BGR2RGB),'Uniform Noise',
          cv2.cvtColor(mean_filtered,cv2.COLOR_BGR2RGB),'Mean Filter'),
         (cv2.cvtColor(sp_noise,cv2.COLOR_BGR2RGB),'Salt & Pepper',
          cv2.cvtColor(median_filtered,cv2.COLOR_BGR2RGB),'Median Filter')]
for r,(d0,t0,d1,t1) in enumerate(pairs):
    axes[r,0].imshow(d0); axes[r,0].set_title(t0); axes[r,0].axis('off')
    axes[r,1].imshow(d1); axes[r,1].set_title(t1); axes[r,1].axis('off')
plt.tight_layout(); plt.show()

### 12. Histogram Equalization

In [ ]:
hist_equalized = cv2.equalizeHist(gray)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1); plt.imshow(gray,cmap='gray'); plt.title('Original Grayscale')
plt.subplot(1,2,2); plt.imshow(hist_equalized,cmap='gray'); plt.title('Histogram Equalized')
plt.show()
for hist,title in [(cv2.calcHist([gray],[0],None,[256],[0,256]),'Before'),
                   (cv2.calcHist([equalized],[0],None,[256],[0,256]),'After')]:
    plt.figure(); plt.bar(range(256),hist.flatten())
    plt.title(f'Histogram {title} Equalization')
    plt.xlabel('Intensity'); plt.ylabel('Pixels'); plt.show()

### 13. Thresholding (Global, Adaptive, Otsu)

In [ ]:
_,global_thresh  = cv2.threshold(img_gray,127,255,cv2.THRESH_BINARY)
adaptive_thresh  = cv2.adaptiveThreshold(img_gray,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,33,2)
_,otsu_thresh    = cv2.threshold(img_gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
plt.figure(figsize=(12,4))
for i,(data,title) in enumerate([(global_thresh,'Global'),(adaptive_thresh,'Adaptive'),(otsu_thresh,'Otsu')]):
    plt.subplot(1,3,i+1); plt.imshow(data,cmap='gray'); plt.title(title+' Threshold'); plt.axis('off')
plt.tight_layout(); plt.show()

### 14. Segmentation — Canny Edge Detection

In [ ]:
img_b = cv2.imread(SAMPLE_IMG_PATH)
edges = cv2.Canny(img_b, 180, 220)
plt.figure(figsize=(8,4))
plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(img_b,cv2.COLOR_BGR2RGB)); plt.title('Original'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(edges,cmap='gray'); plt.title('Canny Edge Detection'); plt.axis('off')
plt.tight_layout(); plt.show()

### 15. Segmentation — Contour Detection

In [ ]:
img    = cv2.imread(SAMPLE_IMG_PATH)
gray_c = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_,thresh = cv2.threshold(gray_c,127,255,cv2.THRESH_BINARY)
contours,_ = cv2.findContours(thresh,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
img_c = img.copy()
cv2.drawContours(img_c,contours,-1,(0,255,0),2)
plt.figure(figsize=(8,4))
plt.subplot(1,2,1); plt.imshow(thresh,cmap='gray'); plt.title('Binary'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(img_c,cv2.COLOR_BGR2RGB)); plt.title('Contours'); plt.axis('off')
plt.tight_layout(); plt.show()

### 16. Segmentation — K-Means Clustering

In [ ]:
img          = cv2.cvtColor(cv2.imread(SAMPLE_IMG_PATH), cv2.COLOR_BGR2RGB)
pixel_values = img.reshape((-1,3)).astype(np.float32)
K            = 2
criteria     = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,100,0.2)
_,labels,centers = cv2.kmeans(pixel_values,K,None,criteria,10,cv2.KMEANS_RANDOM_CENTERS)
centers      = np.uint8(centers)
seg_img      = centers[labels.flatten()].reshape(img.shape)
plt.figure(figsize=(8,4))
plt.subplot(1,2,1); plt.imshow(img); plt.title('Original'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(seg_img); plt.title('K-Means (K=2)'); plt.axis('off')
plt.tight_layout(); plt.show()

### 17. Object Detection — Bounding Boxes & Shape Features

In [ ]:
img  = cv2.imread(SAMPLE_IMG_PATH)
gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray,(33,33),0)
_,thresh = cv2.threshold(blur,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
contours,_ = cv2.findContours(thresh,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
output = img.copy(); features = []; count = 0
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 200: continue
    x,y,w,h = cv2.boundingRect(cnt)
    cv2.rectangle(output,(x,y),(x+w,y+h),(0,255,0),2)
    cv2.putText(output,f'Obj {count+1}',(x,y-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,0,255),2)
    perim = cv2.arcLength(cnt,True); M = cv2.moments(cnt)
    cx = int(M['m10']/M['m00']) if M['m00'] else 0
    cy = int(M['m01']/M['m00']) if M['m00'] else 0
    circ = (4*np.pi*area)/(perim**2+1e-5)
    features.append([count+1,area,perim,cx,cy,circ]); count+=1
plt.imshow(cv2.cvtColor(output,cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
print(f"{'Obj':>4} {'Area':>8} {'Perim':>8} {'cx':>5} {'cy':>5} {'Circ':>8}")
for f in features:
    print(f"{f[0]:>4} {f[1]:>8.1f} {f[2]:>8.1f} {f[3]:>5} {f[4]:>5} {f[5]:>8.3f}")

### 18. Dataset-Wide Analysis — Class Distribution & Sample Grid

In [ ]:
all_classes = sorted(os.listdir(TRAIN_DIR))
counts = {c: len(list(Path(f'{TRAIN_DIR}/{c}').glob('*.jpg'))) for c in all_classes}

# Class distribution
fig,ax = plt.subplots(figsize=(24,5))
ax.bar(range(len(counts)), list(counts.values()), color='steelblue', edgecolor='white')
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(list(counts.keys()), rotation=90, fontsize=5)
ax.set_title(f'Training images per class — {len(all_classes)} classes, {sum(counts.values()):,} total images', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

# Sample grid — 1 image per class (first 50 classes for readability)
show_classes = all_classes[:50]
fig, axes = plt.subplots(5,10,figsize=(20,10))
for i,cls in enumerate(show_classes):
    f   = sorted(Path(f'{TRAIN_DIR}/{cls}').glob('*.jpg'))[0]
    img_s = cv2.cvtColor(cv2.imread(str(f)),cv2.COLOR_BGR2RGB)
    axes[i//10,i%10].imshow(img_s); axes[i//10,i%10].axis('off')
    axes[i//10,i%10].set_title(cls,fontsize=4)
plt.suptitle('Sample images (first 50 classes)', fontsize=12)
plt.tight_layout(); plt.show()

---
# PART 2 — Deep Learning Classification

### 1. Device Setup

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

### 2. Transforms

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

### 3. Load Full Dataset

In [ ]:
dataset_path = TRAIN_DIR

# Check path and list classes
if not os.path.exists(dataset_path):
    print(f"Error: '{dataset_path}' does not exist.")
else:
    subdirs = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path,d))]
    print(f'Found {len(subdirs)} subdirectories (classes) in {dataset_path}')
    for s in subdirs[:10]: print(f'  - {s}')
    print('  ...')

dataset     = datasets.ImageFolder(root=dataset_path, transform=transform)
num_classes = len(dataset.classes)
print(f'\nnum_classes: {num_classes}')
print(f'Total images: {len(dataset):,}')

### 4. Train-Test Split (80-20)

In [ ]:
train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print('Train size:', len(train_dataset))
print('Test size:',  len(test_dataset))

### 5. Training Function

In [ ]:
def train_model(model, epochs=5):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    for epoch in range(epochs):
        model.train(); running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward(); optimizer.step()
            running_loss += loss.item()
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}')
    return model

### 6. Evaluation Function

In [ ]:
def evaluate_model(model):
    model.eval(); all_preds=[]; all_labels=[]
    with torch.no_grad():
        for images, labels in test_loader:
            _,preds = torch.max(model(images.to(device)),1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    print('Test Accuracy:', acc)
    return acc

### 7. Normal CNN (Baseline — From Scratch)

In [ ]:
class NormalCNN(nn.Module):
    def __init__(self, num_classes=250):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

normal_cnn = NormalCNN(num_classes=num_classes)
print('\nTraining Normal CNN (Baseline)...')
normal_cnn = train_model(normal_cnn, epochs=10)
print('Evaluating Normal CNN...')
acc_normal = evaluate_model(normal_cnn)

### 8. CNN — No MaxPooling Variant

In [ ]:
class CNN_NoPooling(nn.Module):
    def __init__(self, num_classes=250):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((7,7)), nn.Flatten(),
            nn.Linear(128*7*7,256), nn.ReLU(), nn.Linear(256,num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

cnn_no_pool = CNN_NoPooling(num_classes=num_classes)
print('\nTraining CNN (No Pooling)...')
cnn_no_pool = train_model(cnn_no_pool, epochs=10)
print('Evaluating...')
acc_nopooling = evaluate_model(cnn_no_pool)

### 9. CNN — With Batch Normalization

In [ ]:
class CNN_BatchNorm(nn.Module):
    def __init__(self, num_classes=250):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),  nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*28*28,256), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(256,num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

cnn_bn = CNN_BatchNorm(num_classes=num_classes)
print('\nTraining CNN (BatchNorm)...')
cnn_bn = train_model(cnn_bn, epochs=10)
print('Evaluating...')
acc_batchnorm = evaluate_model(cnn_bn)

### 10. Feature Map Visualization

In [ ]:
class NormalCNN_Vis(nn.Module):
    def __init__(self, num_classes=250):
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1); self.relu=nn.ReLU(); self.pool=nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32,64,3,padding=1); self.conv3=nn.Conv2d(64,128,3,padding=1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*28*28,256), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(256,num_classes)
        )
    def forward(self, x, visualize=False):
        x = self.relu(self.conv1(x))
        if visualize: return x
        x = self.pool(x)
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        return self.classifier(x)

vis_cnn = NormalCNN_Vis(num_classes=num_classes)
print('Training CNN for visualization (5 epochs)...')
vis_cnn = train_model(vis_cnn, epochs=5)

def visualize_feature_maps(model, dataset):
    model.eval()
    image,label = dataset[0]
    with torch.no_grad():
        fmaps = model(image.unsqueeze(0).to(device), visualize=True).cpu()
    print('Feature map shape:', fmaps.shape)
    fig,axes = plt.subplots(1,8,figsize=(20,5))
    for i in range(8):
        axes[i].imshow(fmaps[0,i],cmap='gray'); axes[i].axis('off'); axes[i].set_title(f'Filter {i+1}')
    plt.show()

visualize_feature_maps(vis_cnn, dataset)
image,_ = dataset[0]
plt.imshow(image.permute(1,2,0).cpu()); plt.title('Original Image'); plt.axis('off'); plt.show()

### 11. AlexNet (Pretrained)

In [ ]:
alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
alexnet.classifier[6] = nn.Linear(alexnet.classifier[6].in_features, num_classes)
print('\nTraining AlexNet...')
alexnet = train_model(alexnet, epochs=5)
print('Evaluating AlexNet...')
acc_alexnet = evaluate_model(alexnet)

### 12. SqueezeNet (Pretrained)

In [ ]:
squeezenet = models.squeezenet1_0(weights=models.SqueezeNet1_0_Weights.DEFAULT)
squeezenet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
squeezenet.num_classes   = num_classes
print('\nTraining SqueezeNet...')
squeezenet = train_model(squeezenet, epochs=5)
print('Evaluating SqueezeNet...')
acc_squeezenet = evaluate_model(squeezenet)

### 13. Model Accuracy Comparison

In [ ]:
results = {
    'Normal CNN':     acc_normal,
    'CNN No Pooling': acc_nopooling,
    'CNN BatchNorm':  acc_batchnorm,
    'AlexNet':        acc_alexnet,
    'SqueezeNet':     acc_squeezenet,
}

print('\n=== FINAL RESULTS ===')
for model_name, acc in results.items():
    print(f'{model_name:<20} {acc*100:.2f}%')

plt.figure(figsize=(10,5))
bars = plt.bar(results.keys(), [v*100 for v in results.values()],
               color='steelblue', edgecolor='white')
plt.title(f'Test Accuracy Comparison — Fruits-360 ({num_classes} classes, {len(dataset):,} images)')
plt.ylabel('Accuracy (%)')
plt.ylim(0,100)
plt.xticks(rotation=15)
for bar,val in zip(bars, results.values()):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{val*100:.1f}%', ha='center', fontsize=9)
plt.tight_layout(); plt.show()